In [172]:
import json
import csv
import re

import fitz  # PyMuPDF
import pandas as pd

from pathlib import Path
from typing import Optional, List, Dict, Any
from dataclasses import dataclass, asdict


from docx import Document
from docx.document import Document as _Document
from docx.table import Table, _Cell
from docx.text.paragraph import Paragraph
from docx.oxml.text.paragraph import CT_P
from docx.oxml.table import CT_Tbl

from tqdm.auto import tqdm

print("All libraries imported successfully.")

All libraries imported successfully.


## Project Path

In [140]:
PROJECT_PATH = Path.cwd().parent

DATA_DIR = PROJECT_PATH / "data"

RAW_DIR = DATA_DIR / "raw"

PDF_DIR = RAW_DIR / "pdf"
DOCX_DIR = RAW_DIR / "docx"
TXT_DIR = RAW_DIR / "txt"
MARKDOWN_DIR = RAW_DIR / "markdown"
CSV_DIR = RAW_DIR / "csv"
EXCEL_DIR = RAW_DIR / "excel"

PROCESSED_DIR = DATA_DIR / "processed"

IMAGE_DIR = PROCESSED_DIR / "images"
CHUNKS_DIR = PROCESSED_DIR / "chunks"
NORMALIZED_DIR = PROCESSED_DIR / "normalized"

VECTOR_STORE_DIR = DATA_DIR / "vector_store"

print("DIR defined!")

DIR defined!


## Normalized schema

In [141]:
@dataclass
class DocumentRecord:
    document: str
    file_type: str
    page: Optional[int] = None
    sheet: Optional[str] = None
    row_start: Optional[int] = None
    row_end: Optional[int] = None
    content_type: str = "text"
    text: str = ""

    # Additional useful metadata
    section: Optional[str] = None
    image_path: Optional[str] = None
    source_path: Optional[str] = None
    table_id: Optional[str] = None

## Convert records to dictionaries

In [142]:
def record_to_dict(record: DocumentRecord) -> Dict[str, any]:
    return asdict(record)

## Clean text utility

In [143]:
def clean_text(text: str) -> str:
    if not text:
        return ""
    
    # Normalize line endings
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # Remove excessive spaces
    text = re.sub(r"[ \t]+", " ", text)

    # Reduce excessive blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

## PDF Loader

In [144]:
def extract_pdf_images(
    doc,
    pdf_path: Path,
    output_dir: Path
) -> List[Dict[str, Any]]:
    
    output_dir.mkdir(parents=True, exist_ok=True)

    image_records = []
    seen_xrefs = set()

    for page_index, page in enumerate(doc):
        page_number = page_index + 1

        for image_index, image_info in enumerate(
            page.get_images(full=True),
            start=1
        ):
            xref = image_info[0]

            # Same embedded image may be referenced multiple times.
            if xref in seen_xrefs:
                continue

            seen_xrefs.add(xref)

            try:
                image_data = doc.extract_image(xref)

                image_bytes = image_data["image"]
                image_ext = image_data["ext"]

                image_filename = (
                    f"{pdf_path.stem}"
                    f"_page_{page_number}"
                    f"_image_{image_index}."
                    f"{image_ext}"
                )

                image_path = output_dir / image_filename

                with open(image_path, "wb") as f:
                    f.write(image_bytes)

                image_records.append({
                    "document": pdf_path.name,
                    "page": page_number,
                    "image_path": str(image_path),
                    "xref": xref,
                    "extension": image_ext
                })

            except Exception as e:
                print(
                    f"Warning: failed to extract image "
                    f"from {pdf_path.name}, page {page_number}: {e}"
                )

    return image_records

### Convert PDF tables into retrieval-friendly text

In [145]:
def dataframe_to_markdown(df: pd.DataFrame) -> str:
    df = df.copy()

    # Replace NaN with empty strings
    df = df.fillna("")

    # Convert column names to strings
    df.columns = [str(col) for col in df.columns]

    return df.to_markdown(index=False)

### PDF table extraction

In [146]:
def extract_pdf_tables(
    page,
    document_name: str,
    page_number: int
) -> List[DocumentRecord]:

    records = []

    try:
        table_finder = page.find_tables()
        tables = table_finder.tables

    except Exception as e:
        print(
            f"Warning: table detection failed on "
            f"{document_name}, page {page_number}: {e}"
        )
        return records

    for table_index, table in enumerate(tables, start=1):

        try:
            df = table.to_pandas()

            if df.empty:
                continue

            table_text = dataframe_to_markdown(df)

            records.append(
                DocumentRecord(
                    document=document_name,
                    file_type="pdf",
                    page=page_number,
                    content_type="table",
                    text=(
                        f"Table {table_index} "
                        f"from page {page_number}\n\n"
                        f"{table_text}"
                    ),
                    table_id=f"{document_name}_"
                             f"page_{page_number}_"
                             f"table_{table_index}"
                )
            )

        except Exception as e:
            print(
                f"Warning: failed to extract table "
                f"{table_index} from {document_name}, "
                f"page {page_number}: {e}"
            )

    return records

### Extract PDF text while avoiding table duplication

In [147]:
def extract_pdf_text(
    page,
    document_name: str,
    page_number: int
) -> List[DocumentRecord]:

    records = []

    try:
        # Get detected tables so their text is not duplicated
        # in the normal page-text record.
        table_finder = page.find_tables()
        tables = table_finder.tables

        table_rects = [
            table.bbox
            for table in tables
        ]

    except Exception:
        table_rects = []

    try:
        blocks = page.get_text("blocks")

        text_parts = []

        for block in blocks:
            x0, y0, x1, y1, text = block[:5]

            if not text or not text.strip():
                continue

            block_rect = fitz.Rect(x0, y0, x1, y1)

            # Skip blocks that overlap detected tables.
            overlaps_table = any(
                block_rect.intersects(fitz.Rect(table_rect))
                for table_rect in table_rects
            )

            if overlaps_table:
                continue

            cleaned = clean_text(text)

            if cleaned:
                text_parts.append(cleaned)

        page_text = "\n\n".join(text_parts)

        if page_text:
            records.append(
                DocumentRecord(
                    document=document_name,
                    file_type="pdf",
                    page=page_number,
                    content_type="text",
                    text=page_text
                )
            )

    except Exception as e:
        print(
            f"Warning: failed to extract text from "
            f"{document_name}, page {page_number}: {e}"
        )

    return records

### Complete PDF Loader

In [148]:
def load_pdf(pdf_path: Path) -> List[DocumentRecord]:

    records = []

    try:
        doc = fitz.open(pdf_path)

        # Handle encrypted/password-protected PDFs
        if doc.needs_pass:
            raise ValueError(
                f"PDF is password protected: {pdf_path.name}"
            )

        document_name = pdf_path.name

        for page_index, page in enumerate(doc):
            page_number = page_index + 1

            # Extract normal text
            text_records = extract_pdf_text(
                page=page,
                document_name=document_name,
                page_number=page_number
            )

            records.extend(text_records)

            # Extract tables
            table_records = extract_pdf_tables(
                page=page,
                document_name=document_name,
                page_number=page_number
            )

            records.extend(table_records)

        doc.close()

    except Exception as e:
        print(
            f"Error loading PDF {pdf_path.name}: {e}"
        )

    return records

### Test PDF Loader

In [149]:
pdf_files = list(PDF_DIR.glob("*.pdf"))

print(f"PDF files found: {len(pdf_files)}")

if pdf_files:
    test_pdf = pdf_files[0]

    pdf_records = load_pdf(test_pdf)

    print(f"Records extracted: {len(pdf_records)}")

    for record in pdf_records[:3]:
        print("\n--------------------")
        print("Document:", record.document)
        print("Page:", record.page)
        print("Content type:", record.content_type)
        print("Text preview:")
        print(record.text[:500])

PDF files found: 0


## DOCX Loader

In [150]:
def iter_docx_blocks(document):
    """
    Yield paragraphs and tables in their original document order.
    """

    body = document.element.body

    for child in body.iterchildren():

        if isinstance(child, CT_P):
            yield Paragraph(child, document)

        elif isinstance(child, CT_Tbl):
            yield Table(child, document)

### DOCX table conversion

In [151]:
def docx_table_to_text(table: Table) -> str:

    rows = table.rows

    if not rows:
        return ""

    output = []

    # Use first row as header
    headers = [
        clean_text(cell.text)
        for cell in rows[0].cells
    ]

    headers = [
        header if header else f"Column {i + 1}"
        for i, header in enumerate(headers)
    ]

    for row_index, row in enumerate(rows[1:], start=1):

        values = [
            clean_text(cell.text)
            for cell in row.cells
        ]

        row_parts = []

        for column_index, value in enumerate(values):

            if column_index < len(headers):
                column_name = headers[column_index]
            else:
                column_name = f"Column {column_index + 1}"

            if value:
                row_parts.append(
                    f"{column_name}: {value}"
                )

        if row_parts:
            output.append(
                f"Row {row_index}: "
                + " | ".join(row_parts)
            )

    return "\n".join(output)

### Final DOCX loader

In [152]:
def load_docx(docx_path: Path) -> List[DocumentRecord]:

    records = []

    try:
        document = Document(docx_path)

        document_name = docx_path.name

        heading_path = []
        paragraph_index = 0
        table_index = 0

        for block in iter_docx_blocks(document):

            # Paragraph
            if isinstance(block, Paragraph):

                text = clean_text(block.text)

                if not text:
                    continue

                paragraph_index += 1

                style_name = ""

                try:
                    style_name = block.style.name or ""
                except Exception:
                    pass

                # Detect headings
                if style_name.lower().startswith("heading"):

                    match = re.search(
                        r"(\d+)",
                        style_name
                    )

                    if match:
                        level = int(match.group(1))

                        heading_path = (
                            heading_path[:level - 1]
                        )

                        heading_path.append(text)

                    continue

                section = " > ".join(heading_path)

                contextual_text = text

                if section:
                    contextual_text = (
                        f"Section: {section}\n\n"
                        f"{text}"
                    )

                records.append(
                    DocumentRecord(
                        document=document_name,
                        file_type="docx",
                        content_type="text",
                        text=contextual_text,
                        section=section or None,
                        source_path=str(docx_path)
                    )
                )

            # Table
            elif isinstance(block, Table):

                table_index += 1

                table_text = docx_table_to_text(block)

                if not table_text:
                    continue

                section = " > ".join(heading_path)

                contextual_text = (
                    f"Table {table_index}"
                )

                if section:
                    contextual_text += (
                        f"\nSection: {section}"
                    )

                contextual_text += (
                    f"\n\n{table_text}"
                )

                records.append(
                    DocumentRecord(
                        document=document_name,
                        file_type="docx",
                        content_type="table",
                        text=contextual_text,
                        section=section or None,
                        table_id=(
                            f"{document_name}_"
                            f"table_{table_index}"
                        ),
                        source_path=str(docx_path)
                    )
                )

    except Exception as e:

        print(
            f"Error loading DOCX "
            f"{docx_path.name}: {e}"
        )

    return records

### DOCX loader test

In [153]:
docx_files = list(DOCX_DIR.glob("*.docx"))

print(f"DOCX files found: {len(docx_files)}")

if docx_files:

    test_docx = docx_files[0]

    docx_records = load_docx(test_docx)

    print(
        f"Records extracted: "
        f"{len(docx_records)}"
    )

    for record in docx_records[:3]:

        print("\n--------------------")
        print("Document:", record.document)
        print("Content type:", record.content_type)
        print("Section:", record.section)
        print("Text:")
        print(record.text[:5000])

DOCX files found: 4
Records extracted: 175

--------------------
Document: 03_Credit_Delegated_Authority_Schedule_CDAS-007_v2.1.docx
Content type: text
Section: None
Text:
NORTHSTAR FINANCIAL

--------------------
Document: 03_Credit_Delegated_Authority_Schedule_CDAS-007_v2.1.docx
Content type: text
Section: None
Text:
Credit Delegated Authority Schedule

--------------------
Document: 03_Credit_Delegated_Authority_Schedule_CDAS-007_v2.1.docx
Content type: text
Section: None
Text:
CDAS-007


## TXT Loader

In [154]:
def load_txt(txt_path: Path) -> List[DocumentRecord]:

    records = []

    try:

        # Try UTF-8 first
        try:
            text = txt_path.read_text(
                encoding="utf-8"
            )

        except UnicodeDecodeError:

            # Fallback for common Windows text files
            text = txt_path.read_text(
                encoding="latin-1"
            )

        text = clean_text(text)

        if text:

            records.append(
                DocumentRecord(
                    document=txt_path.name,
                    file_type="txt",
                    content_type="text",
                    text=text,
                    source_path=str(txt_path)
                )
            )

    except Exception as e:

        print(
            f"Error loading TXT "
            f"{txt_path.name}: {e}"
        )

    return records

## Markdown Loader

In [155]:
def load_markdown(md_path: Path) -> List[DocumentRecord]:

    records = []

    try:

        try:
            text = md_path.read_text(
                encoding="utf-8"
            )

        except UnicodeDecodeError:
            text = md_path.read_text(
                encoding="latin-1"
            )

        lines = text.splitlines()

        heading_path = []
        current_block = []

        def flush_block():

            if not current_block:
                return

            block_text = clean_text(
                "\n".join(current_block)
            )

            if not block_text:
                return

            section = " > ".join(
                heading_path
            )

            contextual_text = block_text

            if section:
                contextual_text = (
                    f"Section: {section}\n\n"
                    f"{block_text}"
                )

            records.append(
                DocumentRecord(
                    document=md_path.name,
                    file_type="markdown",
                    content_type="text",
                    text=contextual_text,
                    section=section or None,
                    source_path=str(md_path)
                )
            )

            current_block.clear()

        for line in lines:

            heading_match = re.match(
                r"^(#{1,6})\s+(.+?)\s*$",
                line
            )

            if heading_match:

                flush_block()

                level = len(
                    heading_match.group(1)
                )

                heading = clean_text(
                    heading_match.group(2)
                )

                heading_path = (
                    heading_path[:level - 1]
                )

                heading_path.append(
                    heading
                )

            elif line.strip() == "":

                flush_block()

            else:

                current_block.append(line)

        flush_block()

    except Exception as e:

        print(
            f"Error loading Markdown "
            f"{md_path.name}: {e}"
        )

    return records

## CSV Loader

In [156]:
def format_structured_row(
    row: pd.Series,
    columns: List[str]
) -> str:

    parts = []

    for column in columns:

        value = row.get(column, "")

        if pd.isna(value):
            value = ""

        value = str(value).strip()

        if value:
            parts.append(
                f"{column}: {value}"
            )

    return "\n".join(parts)

In [157]:
def load_csv(csv_path: Path) -> List[DocumentRecord]:

    records = []

    try:

        df = pd.read_csv(
            csv_path,
            dtype=object
        )

        df = df.fillna("")

        columns = [
            str(column).strip()
            for column in df.columns
        ]

        # Schema record
        schema_text = (
            f"CSV file: {csv_path.name}\n\n"
            f"Columns: "
            + ", ".join(columns)
        )

        records.append(
            DocumentRecord(
                document=csv_path.name,
                file_type="csv",
                content_type="schema",
                text=schema_text,
                row_start=1,
                row_end=1,
                source_path=str(csv_path)
            )
        )

        # Individual rows
        for index, row in df.iterrows():

            row_text = format_structured_row(
                row,
                columns
            )

            if not row_text:
                continue

            # Physical CSV row:
            # row 1 = header
            # row 2 = first data row
            physical_row = index + 2

            records.append(
                DocumentRecord(
                    document=csv_path.name,
                    file_type="csv",
                    content_type="structured_row",
                    text=row_text,
                    row_start=physical_row,
                    row_end=physical_row,
                    source_path=str(csv_path)
                )
            )

    except Exception as e:

        print(
            f"Error loading CSV "
            f"{csv_path.name}: {e}"
        )

    return records

## Excel Loader

In [158]:
def load_excel(excel_path: Path) -> List[DocumentRecord]:

    records = []

    try:

        sheets = pd.read_excel(
            excel_path,
            sheet_name=None,
            dtype=object
        )

        for sheet_name, df in sheets.items():

            if df.empty:
                continue

            df = df.fillna("")

            columns = [
                str(column).strip()
                for column in df.columns
            ]

            # Schema record for this sheet
            schema_text = (
                f"Workbook: {excel_path.name}\n"
                f"Sheet: {sheet_name}\n\n"
                f"Columns: "
                + ", ".join(columns)
            )

            records.append(
                DocumentRecord(
                    document=excel_path.name,
                    file_type="excel",
                    sheet=str(sheet_name),
                    content_type="schema",
                    text=schema_text,
                    row_start=1,
                    row_end=1,
                    source_path=str(excel_path)
                )
            )

            # Individual rows
            for index, row in df.iterrows():

                row_text = format_structured_row(
                    row,
                    columns
                )

                if not row_text:
                    continue

                # Assuming first row is header.
                # Physical Excel row therefore starts at 2.
                physical_row = index + 2

                records.append(
                    DocumentRecord(
                        document=excel_path.name,
                        file_type="excel",
                        sheet=str(sheet_name),
                        content_type="structured_row",
                        text=row_text,
                        row_start=physical_row,
                        row_end=physical_row,
                        source_path=str(excel_path)
                    )
                )

    except Exception as e:

        print(
            f"Error loading Excel "
            f"{excel_path.name}: {e}"
        )

    return records

## Universal Document Loader

In [159]:
LOADER_REGISTRY = {
    ".pdf": load_pdf,
    ".docx": load_docx,
    ".txt": load_txt,
    ".md": load_markdown,
    ".markdown": load_markdown,
    ".csv": load_csv,
    ".xlsx": load_excel,
    ".xls": load_excel,
}

In [160]:
def load_document(file_path: Path) -> List[DocumentRecord]:

    file_path = Path(file_path)

    if not file_path.exists():
        raise FileNotFoundError(
            f"File not found: {file_path}"
        )

    suffix = file_path.suffix.lower()

    if suffix not in LOADER_REGISTRY:
        raise ValueError(
            f"Unsupported file type: {suffix}"
        )

    loader = LOADER_REGISTRY[suffix]

    return loader(file_path)

## Discover all documents

In [161]:
SUPPORTED_EXTENSIONS = set(
    LOADER_REGISTRY.keys()
)

ALL_RAW_DIRS = [
    PDF_DIR,
    DOCX_DIR,
    TXT_DIR,
    MARKDOWN_DIR,
    CSV_DIR,
    EXCEL_DIR,
]

In [162]:
def discover_documents() -> List[Path]:

    files = []

    for directory in ALL_RAW_DIRS:

        if not directory.exists():
            continue

        for file_path in directory.rglob("*"):

            if (
                file_path.is_file()
                and file_path.suffix.lower()
                in SUPPORTED_EXTENSIONS
            ):
                files.append(file_path)

    return sorted(files)

### Test

In [163]:
files = discover_documents()

print(f"Total supported files: {len(files)}")

for file_path in files[:20]:
    print(file_path)

Total supported files: 4
/Users/pushkarkamat/Desktop/financial-rag/data/raw/docx/01_Credit_Risk_Policy_CRP-001_v2.0.docx
/Users/pushkarkamat/Desktop/financial-rag/data/raw/docx/02_Credit_Exception_Procedure_CEP-006_v1.3.docx
/Users/pushkarkamat/Desktop/financial-rag/data/raw/docx/03_Credit_Delegated_Authority_Schedule_CDAS-007_v2.1.docx
/Users/pushkarkamat/Desktop/financial-rag/data/raw/docx/04_Loan_Origination_Policy_LOP-002_v2.3.docx


## Ingest All Documents

In [164]:
all_records = []
ingestion_errors = []

for file_path in files:

    print(f"Processing: {file_path.name}")

    try:

        records = load_document(file_path)

        all_records.extend(records)

        print(
            f"  Records created: {len(records)}"
        )

    except Exception as e:

        error_info = {
            "file": str(file_path),
            "error": str(e)
        }

        ingestion_errors.append(
            error_info
        )

        print(
            f"  ERROR: {e}"
        )

Processing: 01_Credit_Risk_Policy_CRP-001_v2.0.docx
  Records created: 239
Processing: 02_Credit_Exception_Procedure_CEP-006_v1.3.docx
  Records created: 183
Processing: 03_Credit_Delegated_Authority_Schedule_CDAS-007_v2.1.docx
  Records created: 175
Processing: 04_Loan_Origination_Policy_LOP-002_v2.3.docx
  Records created: 271


## Convert records to dictionaries

In [165]:
normalized_documents = [
    record_to_dict(record)
    for record in all_records
]

print(
    f"Total normalized records: "
    f"{len(normalized_documents)}"
)

Total normalized records: 868


### Inspect record types

In [166]:
from collections import Counter

content_type_counts = Counter(
    record["content_type"]
    for record in normalized_documents
)

file_type_counts = Counter(
    record["file_type"]
    for record in normalized_documents
)

print("Content types:")
for key, value in content_type_counts.items():
    print(f"  {key}: {value}")

print("\nFile types:")
for key, value in file_type_counts.items():
    print(f"  {key}: {value}")

Content types:
  text: 840
  table: 28

File types:
  docx: 868


## Preview normalized records

In [167]:
for record in normalized_documents[:5]:

    print("\n" + "=" * 70)

    print(
        "Document:",
        record["document"]
    )

    print(
        "File type:",
        record["file_type"]
    )

    print(
        "Content type:",
        record["content_type"]
    )

    print(
        "Page:",
        record["page"]
    )

    print(
        "Sheet:",
        record["sheet"]
    )

    print(
        "Rows:",
        record["row_start"],
        "-",
        record["row_end"]
    )

    print(
        "\nText preview:\n",
        record["text"][:1000]
    )


Document: 01_Credit_Risk_Policy_CRP-001_v2.0.docx
File type: docx
Content type: text
Page: None
Sheet: None
Rows: None - None

Text preview:
 NORTHSTAR FINANCIAL

Document: 01_Credit_Risk_Policy_CRP-001_v2.0.docx
File type: docx
Content type: text
Page: None
Sheet: None
Rows: None - None

Text preview:
 Credit Risk Policy

Document: 01_Credit_Risk_Policy_CRP-001_v2.0.docx
File type: docx
Content type: text
Page: None
Sheet: None
Rows: None - None

Text preview:
 CRP-001

Document: 01_Credit_Risk_Policy_CRP-001_v2.0.docx
File type: docx
Content type: text
Page: None
Sheet: None
Rows: None - None

Text preview:
 Version 2.0 | Effective 1 April 2025

Document: 01_Credit_Risk_Policy_CRP-001_v2.0.docx
File type: docx
Content type: text
Page: None
Sheet: None
Rows: None - None

Text preview:
 Classification: INTERNAL — CONTROLLED


## Save Normalized Documents

In [168]:
normalized_output = (NORMALIZED_DIR / "normalized_documents.jsonl")

NORMALIZED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

with open(
    normalized_output,
    "w",
    encoding="utf-8"
) as f:

    for record in normalized_documents:

        f.write(
            json.dumps(
                record,
                ensure_ascii=False
            )
            + "\n"
        )

print(
    f"Saved normalized documents to:\n"
    f"{normalized_output}"
)

Saved normalized documents to:
/Users/pushkarkamat/Desktop/financial-rag/data/processed/normalized/normalized_documents.jsonl


## Final Validation 

In [169]:
required_fields = [
    "document",
    "file_type",
    "page",
    "sheet",
    "row_start",
    "row_end",
    "content_type",
    "text",
]

In [170]:
for index, record in enumerate(
    normalized_documents
):

    missing = [
        field
        for field in required_fields
        if field not in record
    ]

    if missing:
        raise ValueError(
            f"Record {index} is missing: "
            f"{missing}"
        )

print(
    "All normalized records passed "
    "schema validation."
)

All normalized records passed schema validation.


In [171]:
empty_text_records = [
    record
    for record in normalized_documents
    if (
        record["content_type"]
        not in {"image"}
        and not record["text"].strip()
    )
]

print(
    "Empty text records:",
    len(empty_text_records)
)

Empty text records: 0
